In [25]:
import openai, json
import os
import requests

from openai.types.chat import ChatCompletionMessage

client = openai.OpenAI()
base_url = os.getenv("MOVIE_API_BASE_URL")
messages = []


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the list of popular movies",
            "parameters": {}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get the details of a movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "string",
                        "description": "The id of the movie"
                    }
                },
                "required": ["movie_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the credits of a movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "string",
                        "description": "The id of the movie"
                    }
                },
                "required": ["movie_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get the similar movies of a movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "string",
                        "description": "The id of the movie"
                    }
                },
            }
        }
    }
]

def get_popular_movies():
    if not base_url:
        raise RuntimeError("MOVIE_API_BASE_URL environment variable not set.")
    url = f"{base_url}/movies"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

def get_movie_details(movie_id):
    if not movie_id:
        raise ValueError("movie_id is required")
    if not base_url:
        raise RuntimeError("MOVIE_API_BASE_URL environment variable not set.")
    url = f"{base_url}/movies/{movie_id}"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

def get_movie_credits(movie_id):
    if not movie_id:
        raise ValueError("movie_id is required")
    if not base_url:
        raise RuntimeError("MOVIE_API_BASE_URL environment variable not set.")
    url = f"{base_url}/movies/{movie_id}/credits"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

def get_similar_movies(movie_id):
    if not movie_id:
        raise ValueError("movie_id is required")
    if not base_url:
        raise RuntimeError("MOVIE_API_BASE_URL environment variable not set.")
    url = f"{base_url}/movies/{movie_id}/similar"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies
}

In [26]:
def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [{
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls]
        })
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            try:
                function_args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError:
                function_args = {}

            print(f"tool calling... {function_name}({function_args})")
            result = FUNCTION_MAP[function_name](**function_args)
            print(f"tool result: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result),
            })
        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")

    
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [27]:
while True:
    message = input("Enter a message: ")
    if message == "exit" or message == "quit" or message == "bye" or message == "q":
        break
    messages.append({"role": "user", "content": message})
    print(f"User: {message}")
    call_ai()

User: 지금 인기있는 영화가 뭐야 ?
tool calling... get_popular_movies({})
tool result: [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/1x9e0qWonw634NhIsRdvnneeqvN.jpg', 'genre_ids': [10749, 18], 'id': 1523145, 'original_language': 'ru', 'original_title': 'Твоё сердце будет разбито', 'overview': 'High school student Polina is saved from bullying at her new school and makes a deal with the main bully Bars: he must pretend to be her boyfriend and protect her, and she must do everything he says. During this game, the couple develops real feelings, but her family and classmates have reasons to separate the lovers.', 'popularity': 929.5163, 'poster_path': 'https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg', 'release_date': '2026-03-26', 'title': 'Your Heart Will Be Broken', 'video': False, 'vote_average': 6.75, 'vote_count': 54}, {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/iN41Ccw4DctL8npfmYg1j5Tr1eb.jpg', 'genre_ids': [878, 12, 14], 'id': 83533